In [1]:
import pandas as pd
import polyline
import requests
import importlib

from main import otp_candidates_df

In [2]:
import load_TU_data
tu_session, tu_tur, tu_deltur = load_TU_data.load_tu(
    data_dir = "~/O/TU_Rejseplan/Data/TU/",
    session_file = "tu_session_secret_2015_2025.xlsx",
    tur_file = "tu_tur_secret_2015_2025.xlsx",
    deltur_file = "tu_deltur_2015_2025.xlsx")

/home/simpal/miniconda3/envs/otp/lib/python3.13/site-packages/pandas/core/arrays/timedeltas.py:1163: RuntimeWarning: invalid value encountered in cast
  int_data = data.astype(np.int64)


In [3]:
tu_tur[(tu_tur["DiaryYear"] == 2024) & (tu_tur["DiaryMonth"] >1 ) & (tu_tur["PtNumBoardings"]>1)]
tu_tur.loc[(tu_tur["DiaryYear"] == 2024) & (tu_tur["DiaryMonth"] >1 ) & (tu_tur["PrimMode"]==32), ["SessionId","TurId"]] #35 (Telebus, Flextrafik) not included.

,SessionId,TurId
262861,505506,2647466
262865,505506,2647470
262898,505517,2647507
262899,505517,2647508
262975,505539,2647592
...,...,...
291224,531166,2728451
291226,531166,2728453
293692,532426,2731309
293694,532426,2731311


In [4]:
i_TurId = 2648141
tu_deltur[tu_deltur["TurId"] == i_TurId]

,SessionId,TurId,Delturnr,StageMode,ModeGroup,StageDrivPass,StageLength,StageWaitMin,StageStartMsm,StageDurationMin,Route,FromStation,ToStation,n_deltur
303109,505753,2648141,1,1,1,NaN,1.0,NaN,410.0,13.0,NaN,NaN,NaN,5
303110,505753,2648141,2,31,120,2.0,4.8,3.0,426.0,15.0,6A,NaN,NaN,5
303111,505753,2648141,3,1,1,NaN,0.1,NaN,441.0,2.0,NaN,NaN,NaN,5
303112,505753,2648141,4,34,110,NaN,7.3,8.0,451.0,12.0,M1,Nørreport,Ørestad,5
303113,505753,2648141,5,1,1,NaN,0.5,NaN,463.0,7.0,NaN,NaN,NaN,5


In [5]:
tu_tur[tu_tur["TurId"] == i_TurId]

,TurId,SessionId,TurNr,TripCount,DepartHH,DepartMM,DepartMSM,ArrivalHH,ArrivalMM,ArrivalMSM,...,WorkplE,WorkplN,Startstedadre,Startstedadrn,date,date_str,depart_dt,depart_dt_str,arrival_dt,arrival_dt_str
263444,2648141,505753,1,1.0,6.0,50.0,410.0,7.0,50.0,470.0,...,725413.0,6170969.0,721829.0,6180105.0,2024-02-09,2024-02-09,2024-02-09 06:50:00+01:00,2024-02-09T06:50:00+0100,2024-02-09 07:50:00+01:00,2024-02-09T07:50:00+0100


In [6]:
tu_deltur.loc[(tu_deltur["TurId"] == i_TurId) & (tu_deltur["StageMode"].isin([31, 32, 33, 34, 37, 41])), "Route"]
      #35 (Telebus, Flextrafik) not included.

303110    6A
303112    M1
Name: Route, dtype: str

In [7]:
tu_deltur.loc[(tu_deltur["TurId"]==i_TurId), "StageMode"]
#Need to split this up into direct (agress/egress) and transit.
#Remove duplicates
#Then map to GraphQL commands

303109     1
303110    31
303111     1
303112    34
303113     1
Name: StageMode, dtype: int64

In [8]:
tu_tur.loc[tu_tur["TurId"] == i_TurId,["orig_lat", "orig_lon", "tiladrlat", "tiladrlon"]]

,orig_lat,orig_lon,tiladrlat,tiladrlon
263444,55.715831,12.531739,55.632249,12.581174


In [9]:
#Transit mode mapping from TU to OTP
mode_map = {
#   TU: OTP
    31: "BUS",
    32: "S_TRAIN", #"S_TRAIN" added to OTP by Simun. Originally classified as RAIL
    33: "RAIL",
    34: "SUBWAY",
    37: "TRAM",
    41: "FERRY",
    35: "BUS" #Maybe remove trips with 35 #35: Telebus, Flextrafik - Behovsstyrede kollektive trafik #Often basically a taxi
}

In [10]:
modes_list = (
    tu_deltur.loc[
        (tu_deltur["TurId"]==i_TurId) &
        (tu_deltur["StageMode"].isin([31, 32, 33, 34, 37, 41])),
        "StageMode"
    ]
    .drop_duplicates()
    .map(mode_map)
    .tolist()
)
modes_json = [{"mode": m} for m in modes_list]
modes_json

[{'mode': 'BUS'}, {'mode': 'SUBWAY'}]

In [11]:
tu_deltur["Route"] = tu_deltur["Route"].astype(str)
route_short_name = tu_deltur.loc[
    (tu_deltur["TurId"]==i_TurId) &
    (tu_deltur["StageMode"].isin([31, 32, 33, 34, 37, 41])),
    "Route"
].drop_duplicates().astype(str).tolist()
print(route_short_name)

['6A', 'M1']


In [ ]:
tu_tur.loc[tu_tur["TurId"]==i_TurId, "depart_dt"]

In [50]:
import otp_client
importlib.reload(otp_client)
from otp_client import get_all_routes_for_mode
# RAIL, TRAM and SUBWAY are missing route name in TU.
# So taking all routes for these modes. Which will be used when modes
# that do include route name in TU only can access those routes, but for
# those that do not, all routes will be used.
otp_mode_routes_cache = {}
otp_mode_routes_cache["RAIL"] = get_all_routes_for_mode("RAIL")
otp_mode_routes_cache["TRAM"] = get_all_routes_for_mode("TRAM")
otp_mode_routes_cache["SUBWAY"] = get_all_routes_for_mode("SUBWAY")

In [51]:
otp_mode_routes_cache

{'RAIL': ['510R',
  '920R',
  'RX',
  '030',
  '802',
  '031',
  '93',
  '920E',
  'IL',
  '806',
  '805',
  'IC',
  '930R',
  'RX',
  '803',
  '76',
  '710R',
  'RE76',
  '92',
  '005',
  '030',
  'EC',
  '940R',
  '804',
  'RE75',
  '75',
  '69',
  '93Tog',
  '210R',
  'RE',
  '950R',
  '005',
  'RE69',
  '031',
  '110R',
  '910',
  '410',
  '960R',
  'ICL',
  '92Tog'],
 'TRAM': ['L', 'L1', 'L2'],
 'SUBWAY': ['M4', 'M2', 'M3', 'M1']}

In [38]:
import otp_client
importlib.reload(otp_client)
from otp_client import graphql_json_request
import otp_parser
importlib.reload(otp_parser)
from otp_parser import json_to_df

tu_row = tu_tur.loc[tu_tur["TurId"] == i_TurId].iloc[0]

search_window = "PT10M"
resp_depart_dt = tu_tur.loc[tu_tur["TurId"]==i_TurId, "depart_dt"].iloc[0]
#Inital request
response = graphql_json_request(tu_row=tu_row, modes_json=modes_json, route_short_name_json=route_short_name,
                                direct=["WALK"], first=50, direct_only=False, transit_only=True,
                                search_window=search_window, url="http://localhost:8080/otp/gtfs/v1")
otp_candidates_df = json_to_df(response)
response_data = response.json()


In [39]:
n_forward = len(response_data["data"]["planConnection"]["edges"])
hasNextPage = response_data["data"]["planConnection"]["pageInfo"]["hasNextPage"]
while n_forward < 50 and hasNextPage:
    # Check if all trips are within the search window
    otp_candidates_df["start_dt"] = pd.to_datetime(otp_candidates_df["start"]).dt.tz_convert('Europe/Copenhagen')
    trips_within_window = ((otp_candidates_df["start_dt"] - resp_depart_dt) <= pd.Timedelta(search_window)).all()
    if not trips_within_window:
        break

    endCursor = response_data["data"]["planConnection"]["pageInfo"]["endCursor"]
    response = graphql_json_request(tu_row=tu_row, modes_json=modes_json, route_short_name_json=route_short_name,
                                    direct=["WALK"], after=endCursor, first=50 - n_forward, direct_only=False,
                                    transit_only=True, search_window=search_window,
                                    url="http://localhost:8080/otp/gtfs/v1")
    otp_trip_candidates_forward = json_to_df(response)
    #Adjust iteration_id before concatenating
    offset = otp_candidates_df['iteration_id'].max() + 1
    otp_trip_candidates_forward['iteration_id'] = otp_trip_candidates_forward['iteration_id'] + offset
    otp_candidates_df = pd.concat([otp_candidates_df, otp_trip_candidates_forward]).reset_index(drop=True)

    response_data = response.json()
    hasNextPage = response_data["data"]["planConnection"]["pageInfo"]["hasNextPage"]
    n_forward += len(response_data["data"]["planConnection"]["edges"])


n_backward = 0
hasPreviousPage = response_data["data"]["planConnection"]["pageInfo"]["hasPreviousPage"]
while n_backward < 50 and hasPreviousPage:
        # Check if all trips are within the search window
    otp_candidates_df["start_dt"] = pd.to_datetime(otp_candidates_df["start"]).dt.tz_convert('Europe/Copenhagen')
    trips_within_window = ((otp_candidates_df["start_dt"] - resp_depart_dt) >= -pd.Timedelta(search_window)).all()
    if not trips_within_window:
        break

    startCursor = response_data["data"]["planConnection"]["pageInfo"]["startCursor"]
    response = graphql_json_request(tu_row=tu_row, modes_json=modes_json, route_short_name_json=route_short_name,
                                    direct=["WALK"], before=startCursor, last=50 - n_backward, direct_only=False,
                                    transit_only=True, search_window=search_window,
                                    url="http://localhost:8080/otp/gtfs/v1")
    otp_trip_candidates_backward = json_to_df(response)
    #Adjust iteration_id before concatenating
    offset = otp_candidates_df['iteration_id'].min() - otp_trip_candidates_backward['iteration_id'].max() - 1
    otp_trip_candidates_backward['iteration_id'] = otp_trip_candidates_backward['iteration_id'] + offset
    otp_candidates_df = pd.concat([otp_trip_candidates_backward, otp_candidates_df]).reset_index(drop=True)

    response_data = response.json()
    hasPreviousPage = response_data["data"]["planConnection"]["pageInfo"]["hasPreviousPage"]
    n_backward += len(response_data["data"]["planConnection"]["edges"])


In [40]:
otp_candidates_df

,start,end,system_notice_tag,system_notice_text,iteration_id,leg_id,mode,route_short_name,distance_km,duration_min,generalized_cost,start_time,end_time,from,to,leg_geometry,start_dt
0,2024-02-09T06:18:30+01:00,2024-02-09T07:03:47+01:00,[],[],-8,0,WALK,NaN,0.031,0,74,1707455910000,1707455940000,Origin,Bispebjerg Torv (Tagensvej),"[(55.71588, 12.53181), (55.71593, 12.53171), (...",2024-02-09 06:18:30+01:00
1,2024-02-09T06:18:30+01:00,2024-02-09T07:03:47+01:00,[],[],-8,1,BUS,6A,4.879,15,1560,1707455940000,1707456840000,Bispebjerg Torv (Tagensvej),Nørreport St. (Nørre Voldgade),"[(55.71592, 12.53195), (55.71592, 12.53195), (...",2024-02-09 06:18:30+01:00
2,2024-02-09T06:18:30+01:00,2024-02-09T07:03:47+01:00,[],[],-8,2,WALK,NaN,0.135,4,381,1707456900000,1707457169000,Nørreport St. (Nørre Voldgade),Nørreport St. (Metro),"[(55.68404, 12.57262), (55.68403, 12.57264), (...",2024-02-09 06:18:30+01:00
3,2024-02-09T06:18:30+01:00,2024-02-09T07:03:47+01:00,[],[],-8,3,SUBWAY,M1,7.241,12,1531,1707457320000,1707458040000,Nørreport St. (Metro),Ørestad St. (Metro),"[(55.68385, 12.57106), (55.68365, 12.57158), (...",2024-02-09 06:18:30+01:00
4,2024-02-09T06:18:30+01:00,2024-02-09T07:03:47+01:00,[],[],-8,4,WALK,NaN,0.447,9,991,1707458100000,1707458627000,Ørestad St. (Metro),Destination,"[(55.62905, 12.57938), (55.62904, 12.57943), (...",2024-02-09 06:18:30+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,2024-02-09T07:01:30+01:00,2024-02-09T07:49:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,4,0,WALK,NaN,0.031,0,74,1707458490000,1707458520000,Origin,Bispebjerg Torv (Tagensvej),"[(55.71588, 12.53181), (55.71593, 12.53171), (...",2024-02-09 07:01:30+01:00
57,2024-02-09T07:01:30+01:00,2024-02-09T07:49:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,4,1,BUS,6A,4.879,18,1740,1707458520000,1707459600000,Bispebjerg Torv (Tagensvej),Nørreport St. (Nørre Voldgade),"[(55.71592, 12.53195), (55.71592, 12.53195), (...",2024-02-09 07:01:30+01:00
58,2024-02-09T07:01:30+01:00,2024-02-09T07:49:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,4,2,WALK,NaN,0.135,4,381,1707459660000,1707459929000,Nørreport St. (Nørre Voldgade),Nørreport St. (Metro),"[(55.68404, 12.57262), (55.68403, 12.57264), (...",2024-02-09 07:01:30+01:00
59,2024-02-09T07:01:30+01:00,2024-02-09T07:49:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,4,3,SUBWAY,M1,7.241,12,1531,1707460080000,1707460800000,Nørreport St. (Metro),Ørestad St. (Metro),"[(55.68385, 12.57106), (55.68365, 12.57158), (...",2024-02-09 07:01:30+01:00


In [41]:
required_routes = set(route_short_name)
type(required_routes)

set

In [42]:
iteration_ids_with_all_routes = (
    otp_candidates_df.groupby("iteration_id")["route_short_name"]
    .apply(lambda routes: required_routes.issubset(set(routes.astype(str))))
)
iteration_ids_with_all_routes

iteration_id
-8     True
-7     True
-6     True
-5     True
-4     True
-3     True
-2     True
-1     True
 0    False
 1     True
 2     True
 3     True
 4     True
Name: route_short_name, dtype: bool

In [44]:
otp_candidates_df = otp_candidates_df[
    otp_candidates_df["iteration_id"].isin(
        iteration_ids_with_all_routes[iteration_ids_with_all_routes].index
    )
].reset_index(drop=True)
otp_candidates_df

,start,end,system_notice_tag,system_notice_text,iteration_id,leg_id,mode,route_short_name,distance_km,duration_min,generalized_cost,start_time,end_time,from,to,leg_geometry,start_dt
0,2024-02-09T06:18:30+01:00,2024-02-09T07:03:47+01:00,[],[],-8,0,WALK,NaN,0.031,0,74,1707455910000,1707455940000,Origin,Bispebjerg Torv (Tagensvej),"[(55.71588, 12.53181), (55.71593, 12.53171), (...",2024-02-09 06:18:30+01:00
1,2024-02-09T06:18:30+01:00,2024-02-09T07:03:47+01:00,[],[],-8,1,BUS,6A,4.879,15,1560,1707455940000,1707456840000,Bispebjerg Torv (Tagensvej),Nørreport St. (Nørre Voldgade),"[(55.71592, 12.53195), (55.71592, 12.53195), (...",2024-02-09 06:18:30+01:00
2,2024-02-09T06:18:30+01:00,2024-02-09T07:03:47+01:00,[],[],-8,2,WALK,NaN,0.135,4,381,1707456900000,1707457169000,Nørreport St. (Nørre Voldgade),Nørreport St. (Metro),"[(55.68404, 12.57262), (55.68403, 12.57264), (...",2024-02-09 06:18:30+01:00
3,2024-02-09T06:18:30+01:00,2024-02-09T07:03:47+01:00,[],[],-8,3,SUBWAY,M1,7.241,12,1531,1707457320000,1707458040000,Nørreport St. (Metro),Ørestad St. (Metro),"[(55.68385, 12.57106), (55.68365, 12.57158), (...",2024-02-09 06:18:30+01:00
4,2024-02-09T06:18:30+01:00,2024-02-09T07:03:47+01:00,[],[],-8,4,WALK,NaN,0.447,9,991,1707458100000,1707458627000,Ørestad St. (Metro),Destination,"[(55.62905, 12.57938), (55.62904, 12.57943), (...",2024-02-09 06:18:30+01:00
5,2024-02-09T06:23:30+01:00,2024-02-09T07:09:47+01:00,[],[],-7,0,WALK,NaN,0.031,0,74,1707456210000,1707456240000,Origin,Bispebjerg Torv (Tagensvej),"[(55.71588, 12.53181), (55.71593, 12.53171), (...",2024-02-09 06:23:30+01:00
6,2024-02-09T06:23:30+01:00,2024-02-09T07:09:47+01:00,[],[],-7,1,BUS,6A,4.879,15,1560,1707456240000,1707457140000,Bispebjerg Torv (Tagensvej),Nørreport St. (Nørre Voldgade),"[(55.71592, 12.53195), (55.71592, 12.53195), (...",2024-02-09 06:23:30+01:00
7,2024-02-09T06:23:30+01:00,2024-02-09T07:09:47+01:00,[],[],-7,2,WALK,NaN,0.135,4,381,1707457200000,1707457469000,Nørreport St. (Nørre Voldgade),Nørreport St. (Metro),"[(55.68404, 12.57262), (55.68403, 12.57264), (...",2024-02-09 06:23:30+01:00
8,2024-02-09T06:23:30+01:00,2024-02-09T07:09:47+01:00,[],[],-7,3,SUBWAY,M1,7.241,12,1591,1707457680000,1707458400000,Nørreport St. (Metro),Ørestad St. (Metro),"[(55.68385, 12.57106), (55.68365, 12.57158), (...",2024-02-09 06:23:30+01:00
9,2024-02-09T06:23:30+01:00,2024-02-09T07:09:47+01:00,[],[],-7,4,WALK,NaN,0.447,9,991,1707458460000,1707458987000,Ørestad St. (Metro),Destination,"[(55.62905, 12.57938), (55.62904, 12.57943), (...",2024-02-09 06:23:30+01:00


In [ ]:
#Calcuate the total deviation OTP trip and TU tur and find best matching trip
import find_similar_trip
importlib.reload(find_similar_trip)
from find_similar_trip import find_similar_trip
best_trip_candidate = find_similar_trip(tu_row, otp_candidates_df, arrival_dev_weight = 1)

In [28]:
best_trip_candidate["TurId"] = i_TurId
best_trip_candidate

,start,end,system_notice_tag,system_notice_text,iteration_id,leg_id,mode,route,distance_km,duration_min,generalized_cost,start_time,end_time,from,to,leg_geometry,start_dt,end_dt,TurId
56,2024-02-09T07:01:30+01:00,2024-02-09T07:49:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,4,0,WALK,NaN,0.031,0,74,1707458490000,1707458520000,Origin,Bispebjerg Torv (Tagensvej),"[(55.71588, 12.53181), (55.71593, 12.53171), (...",2024-02-09 06:01:30+00:00,2024-02-09 06:49:47+00:00,2648141
57,2024-02-09T07:01:30+01:00,2024-02-09T07:49:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,4,1,BUS,6A,4.879,18,1740,1707458520000,1707459600000,Bispebjerg Torv (Tagensvej),Nørreport St. (Nørre Voldgade),"[(55.71592, 12.53195), (55.71592, 12.53195), (...",2024-02-09 06:01:30+00:00,2024-02-09 06:49:47+00:00,2648141
58,2024-02-09T07:01:30+01:00,2024-02-09T07:49:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,4,2,WALK,NaN,0.135,4,381,1707459660000,1707459929000,Nørreport St. (Nørre Voldgade),Nørreport St. (Metro),"[(55.68404, 12.57262), (55.68403, 12.57264), (...",2024-02-09 06:01:30+00:00,2024-02-09 06:49:47+00:00,2648141
59,2024-02-09T07:01:30+01:00,2024-02-09T07:49:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,4,3,SUBWAY,M1,7.241,12,1531,1707460080000,1707460800000,Nørreport St. (Metro),Ørestad St. (Metro),"[(55.68385, 12.57106), (55.68365, 12.57158), (...",2024-02-09 06:01:30+00:00,2024-02-09 06:49:47+00:00,2648141
60,2024-02-09T07:01:30+01:00,2024-02-09T07:49:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,4,4,WALK,NaN,0.447,9,991,1707460860000,1707461387000,Ørestad St. (Metro),Destination,"[(55.62905, 12.57938), (55.62904, 12.57943), (...",2024-02-09 06:01:30+00:00,2024-02-09 06:49:47+00:00,2648141


In [84]:
import folium
import os

def plot_iteration(otp_trip_candidates, iteration_id, file_path=None, zoom_start=12):
    mode_colors = {
        "WALK": "red",
        "BUS": "blue",
        "RAIL": "green",
        "S_TRAIN": "green",
        "SUBWAY": "yellow",
    }

    trip_legs = (
        otp_trip_candidates.loc[otp_trip_candidates["iteration_id"] == iteration_id]
        .sort_values("leg_id")
        .copy()
    )

    if trip_legs.empty:
        raise ValueError(f"No legs found for iteration_id={iteration_id}")

    first_geometry = trip_legs["leg_geometry_length"].dropna().iloc[0]
    map_center = first_geometry[0]

    m = folium.Map(
        location=map_center,
        zoom_start=zoom_start,
        tiles="OpenStreetMap"
    )

    for _, leg in trip_legs.iterrows():
        mode = leg["mode"]
        geometry = leg["leg_geometry_length"]
        color = mode_colors.get(mode, "gray")

        folium.PolyLine(
            geometry,
            color=color,
            weight=5,
            opacity=0.8,
            tooltip=f"{mode} | leg_id={leg['leg_id']} | route={leg.get('route')}"
        ).add_to(m)

    if file_path is not None:
        file_path = os.path.expanduser(file_path)
        m.save(file_path)

    return m

In [75]:
plot_iteration(otp_candidates_df, iteration_id=44)